# E8 — Full SDFL Stack (Final Experiment)

This notebook runs the complete SDFL experiment stack (including differential privacy, secure aggregation, temporal context, sanitized collator, and MC Dropout) on a Google Colab T4 GPU runtime.

### Cell 1 — Verify T4 GPU

In [1]:
import torch
assert torch.cuda.is_available(), "Switch runtime to T4 GPU first"
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Tesla T4
VRAM: 15.6 GB


### Cell 2 — Clone and pull latest repo

In [2]:
!git clone https://github.com/sanjayrk2007/SDFL.git 2>/dev/null || (cd SDFL && git pull origin main)
%cd SDFL
!git log --oneline -5

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 424 bytes | 84.00 KiB/s, done.
From https://github.com/sanjayrk2007/SDFL
 * branch            main       -> FETCH_HEAD
   ffd808e..b0cf95e  main       -> origin/main
Updating ffd808e..b0cf95e
Fast-forward
 e8_server.py | 7 +++++--
 1 file changed, 5 insertions(+), 2 deletions(-)
/content/SDFL
b0cf95e (HEAD -> main, origin/main, origin/HEAD) E8: Fix TypeError on trainloader.batch_size in Opacus wrapper
ffd808e E8: Fix cumulative epsilon tracking and Failure Detection AUC metrics
69e829c E8: Full SDFL stack - weighted aggregation, MC Dropout, cross-centre eval
039723f Update audit_log.jsonl with clean 3-round verification logs
a928bfa Update E7_RESULTS.md with verified 3-round metrics


### Cell 3 — Install dependencies

In [3]:
!pip install -q flwr[simulation] opacus cryptography torch torchvision opencv-python-headless Pillow numpy scipy scikit-learn
!pip install -q medpy || echo "medpy install failed — will use scipy fallback"

### Cell 4 — Download Kvasir-SEG dataset

In [4]:
!unzip -o sdfl_dataset_config.zip -d ./
!python scripts/setup_data.py

import os
print(f"Images: {len(os.listdir('data/images'))}")
print(f"Masks:  {len(os.listdir('data/masks'))}")

Archive:  sdfl_dataset_config.zip
  inflating: ./splits.json           
  inflating: ./hospital_splits.json  
  inflating: ./config.py             
  inflating: ./scripts/visual_check.py  
  inflating: ./scripts/__pycache__/dataset.cpython-312.pyc  
  inflating: ./scripts/__pycache__/joint_transforms.cpython-312.pyc  
  inflating: ./scripts/setup_data.py  
  inflating: ./scripts/joint_transforms.py  
  inflating: ./scripts/make_splits.py  
  inflating: ./scripts/dataset.py    
data/images already populated (1000 files found) — skipping download. Delete the folder contents to force a re-download.
Images: 1000
Masks:  1000


### Cell 5 — Mount Google Drive for checkpoint persistence

In [5]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/SDFL_checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/SDFL_results', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Cell 6 — Verify all upstream checkpoints exist

In [6]:
import os
required = [
    "checkpoints/e7_best.pth",
    "checkpoints/e6_best.pth",
    "checkpoints/e5_best.pth"
]
for path in required:
    status = "✅" if os.path.exists(path) else "❌ MISSING"
    print(f"{status} {path}")

# If e7_best.pth is missing, copy from Google Drive:
if not os.path.exists("checkpoints/e7_best.pth"):
    import shutil
    drive_path = "/content/drive/MyDrive/SDFL_checkpoints/e7_best.pth"
    if os.path.exists(drive_path):
        shutil.copy(drive_path, "checkpoints/e7_best.pth")
        print("Copied e7_best.pth from Drive")

✅ checkpoints/e7_best.pth
✅ checkpoints/e6_best.pth
✅ checkpoints/e5_best.pth


### Cell 7 — Run E8 simulation (20 rounds)

In [7]:
import sys
sys.path.insert(0, '/content/SDFL')
sys.path.insert(0, '/content/SDFL/scripts')

from e8_server import run_e8_simulation
metrics = run_e8_simulation(num_rounds=20)

=== PART 9: Running E8 Federated Learning Simulation ===


	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

          

Round  1/20 | val_loss 0.5155 | val_dice 0.3945 | val_iou 0.2819


INFO :      aggregate_fit: received 3 results and 0 failures
INFO :      configure_evaluate: strategy sampled 3 clients (out of 3)
INFO :      aggregate_evaluate: received 3 results and 0 failures
INFO :      
INFO :      [ROUND 3]
INFO :      configure_fit: strategy sampled 3 clients (out of 3)


Round  2/20 | val_loss 0.5139 | val_dice 0.4030 | val_iou 0.2889


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Saved best model to checkpoints/e8_final.pth
=== Running evaluations ===


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=22177) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


Metrics successfully saved to results/e8_metrics.json

                    E8 - FULL SDFL STACK EXPERIMENT METRICS                     
SEGMENTATION PERFORMANCE:
  In-Distribution (H0 + H1 Test):
    Dice Score:      0.4145
    IoU Score:       0.2994
    Precision:       0.4405
    Recall:          0.5267
    F2 Score:        0.5962
    HD95 Distance:   84.1699
  Out-of-Distribution (H2 Test):
    Dice Score:      0.4869
    IoU Score:       0.3477
    Precision:       0.6161
    Recall:          0.5038
    F2 Score:        0.5663
    HD95 Distance:   78.6602
  Generalisation Gap:
    Dice Gap:        -0.0724
    IoU Gap:         -0.0483
--------------------------------------------------------------------------------
PRIVACY GUARANTEES:
  Final Epsilon:     2.7720 (at delta = 1e-05)
  Post-Expiry Decrypt Success Rate: 0.0% (target: 0.0%)
--------------------------------------------------------------------------------
SYSTEM METRICS (Per-Round Average):
  Encryption Time:   0.0386 seco

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### Cell 8 — Save all outputs to Google Drive

In [8]:
import shutil, json

# Checkpoint
shutil.copy(
    '/content/SDFL/checkpoints/e8_final.pth',
    '/content/drive/MyDrive/SDFL_checkpoints/e8_final.pth'
)

# Metrics JSON
shutil.copy(
    '/content/SDFL/results/e8_metrics.json',
    '/content/drive/MyDrive/SDFL_results/e8_metrics.json'
)

# Audit log
shutil.copy(
    '/content/SDFL/audit_log.jsonl',
    '/content/drive/MyDrive/SDFL_results/audit_log_e8.jsonl'
)

print("All E8 outputs saved to Google Drive")

All E8 outputs saved to Google Drive


### Cell 9 — Print final results summary

In [9]:
import json
with open('/content/SDFL/results/e8_metrics.json') as f:
    m = json.load(f)

print("=" * 50)
print("E8 FULL SDFL STACK — FINAL RESULTS")
print("=" * 50)
print("\nSEGMENTATION")
print(f"  In-distribution Dice : {m['segmentation']['in_distribution']['dice']:.4f}")
print(f"  OOD Dice             : {m['segmentation']['ood']['dice']:.4f}")
print(f"  Generalisation gap   : {m['segmentation']['generalisation_gap']['dice_gap']:.4f}")
print(f"  HD95                 : {m['segmentation']['ood']['hd95']:.2f}")
print("\nPRIVACY")
print(f"  Final ε              : {m['privacy']['final_epsilon']:.4f}")
print(f"  δ                    : {m['privacy']['delta']}")
print(f"  Post-expiry decrypt  : {m['privacy']['post_expiry_decryption_success_rate']*100:.1f}%")
print("\nUNCERTAINTY")
print(f"  ECE                  : {m['uncertainty']['ece']:.4f}")
print(f"  Failure Detection AUC: {m['uncertainty']['failure_detection_auc']:.4f}")
print("\nSYSTEM")

# Safeguard in case system fields are lists
enc_time = sum(m['system']['encryption_time_per_round']) / len(m['system']['encryption_time_per_round']) if isinstance(m['system']['encryption_time_per_round'], list) else m['system']['encryption_time_per_round']
agg_time = sum(m['system']['aggregation_time_per_round']) / len(m['system']['aggregation_time_per_round']) if isinstance(m['system']['aggregation_time_per_round'], list) else m['system']['aggregation_time_per_round']
comm_vol = int(sum(m['system']['communication_bytes_per_round']) / len(m['system']['communication_bytes_per_round'])) if isinstance(m['system']['communication_bytes_per_round'], list) else m['system']['communication_bytes_per_round']

print(f"  Encryption time/round: {enc_time:.3f}s")
print(f"  Aggregation time/round:{agg_time:.3f}s")
print(f"  Comm bytes/round     : {comm_vol:,} bytes")
print("=" * 50)

E8 FULL SDFL STACK — FINAL RESULTS

SEGMENTATION
  In-distribution Dice : 0.4145
  OOD Dice             : 0.4869
  Generalisation gap   : -0.0724
  HD95                 : 78.66

PRIVACY
  Final ε              : 2.7720
  δ                    : 1e-05
  Post-expiry decrypt  : 0.0%

UNCERTAINTY
  ECE                  : 0.0888
  Failure Detection AUC: 0.5052

SYSTEM
  Encryption time/round: 0.039s
  Aggregation time/round:0.083s
  Comm bytes/round     : 79,111,638 bytes


### Cell 10 — Verify audit log integrity

In [10]:
import json

with open('/content/SDFL/audit_log.jsonl') as f:
    lines = [json.loads(l) for l in f]

round_ids = sorted(set(l["round_id"] for l in lines))
print(f"Total log entries : {len(lines)}")
print(f"Rounds logged     : {round_ids}")

for rid in round_ids:
    events = [l["event"] for l in lines if l["round_id"] == rid]
    status = "✅" if set(events) >= {"round_open","round_close","key_destroyed"} else "❌"
    print(f"  Round {rid}: {events} {status}")

Total log entries : 60
Rounds logged     : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
  Round 1: ['round_open', 'round_close', 'key_destroyed'] ✅
  Round 2: ['round_open', 'round_close', 'key_destroyed'] ✅
  Round 3: ['round_open', 'round_close', 'key_destroyed'] ✅
  Round 4: ['round_open', 'round_close', 'key_destroyed'] ✅
  Round 5: ['round_open', 'round_close', 'key_destroyed'] ✅
  Round 6: ['round_open', 'round_close', 'key_destroyed'] ✅
  Round 7: ['round_open', 'round_close', 'key_destroyed'] ✅
  Round 8: ['round_open', 'round_close', 'key_destroyed'] ✅
  Round 9: ['round_open', 'round_close', 'key_destroyed'] ✅
  Round 10: ['round_open', 'round_close', 'key_destroyed'] ✅
  Round 11: ['round_open', 'round_close', 'key_destroyed'] ✅
  Round 12: ['round_open', 'round_close', 'key_destroyed'] ✅
  Round 13: ['round_open', 'round_close', 'key_destroyed'] ✅
  Round 14: ['round_open', 'round_close', 'key_destroyed'] ✅
  Round 15: ['round_open', 'round_close'

In [11]:
import json
import shutil

with open('/content/SDFL/results/e8_metrics.json') as f:
    m = json.load(f)

enc_time = sum(m['system']['encryption_time_per_round']) / len(m['system']['encryption_time_per_round']) if isinstance(m['system']['encryption_time_per_round'], list) else m['system']['encryption_time_per_round']
agg_time = sum(m['system']['aggregation_time_per_round']) / len(m['system']['aggregation_time_per_round']) if isinstance(m['system']['aggregation_time_per_round'], list) else m['system']['aggregation_time_per_round']
comm_vol = int(sum(m['system']['communication_bytes_per_round']) / len(m['system']['communication_bytes_per_round'])) if isinstance(m['system']['communication_bytes_per_round'], list) else m['system']['communication_bytes_per_round']

md_content = f"""# E8 — Full SDFL Stack Experiment Results

This report documents the final metrics of the E8 experiment, evaluating the full SDFL (Secure and Private Federated Learning) stack.

## 1. Segmentation Performance

| Metric | In-Distribution (H0 + H1 Test) | Out-of-Distribution (H2 Test) | Generalisation Gap |
| :--- | :---: | :---: | :---: |
| **Dice Score** | {m['segmentation']['in_distribution']['dice']:.4f} | {m['segmentation']['ood']['dice']:.4f} | {m['segmentation']['generalisation_gap']['dice_gap']:.4f} |
| **IoU Score** | {m['segmentation']['in_distribution']['iou']:.4f} | {m['segmentation']['ood']['iou']:.4f} | {m['segmentation']['generalisation_gap']['iou_gap']:.4f} |
| **Precision** | {m['segmentation']['in_distribution']['precision']:.4f} | {m['segmentation']['ood']['precision']:.4f} | - |
| **Recall** | {m['segmentation']['in_distribution']['recall']:.4f} | {m['segmentation']['ood']['recall']:.4f} | - |
| **F2 Score** | {m['segmentation']['in_distribution']['f2']:.4f} | {m['segmentation']['ood']['f2']:.4f} | - |
| **HD95 (px)** | {m['segmentation']['in_distribution']['hd95']:.2f} | {m['segmentation']['ood']['hd95']:.2f} | - |

> [!NOTE]
> The negative generalization gap (where OOD performance exceeds in-distribution performance) is due to the larger, well-defined polyps contained in the Hospital 2 cohort compared to the smaller, more challenging cases in Hospitals 0 and 1.

## 2. Privacy Guarantees

* **Cumulative Differential Privacy budget (ε)**: `{m['privacy']['final_epsilon']:.4f}` (at $\delta = {m['privacy']['delta']}$)
* **Post-Expiry Decryption Success Rate**: `{m['privacy']['post_expiry_decryption_success_rate'] * 100:.1f}%` (Target: 0.0%)

## 3. Uncertainty & Failure Detection

* **Expected Calibration Error (ECE)**: `{m['uncertainty']['ece']:.4f}` (pixel-level)
* **Failure Detection ROC AUC**: `{m['uncertainty']['failure_detection_auc']:.4f}`

## 4. System Overhead (Per-Round Averages)

* **Client Encryption Time**: `{enc_time:.4f} seconds`
* **Server Aggregation Time**: `{agg_time:.4f} seconds`
* **Communication Volume**: `{comm_vol:,} bytes`
"""

with open('/content/SDFL/E8_RESULTS.md', 'w') as f:
    f.write(md_content)

shutil.copy('/content/SDFL/E8_RESULTS.md', '/content/drive/MyDrive/SDFL_results/E8_RESULTS.md')
print("E8_RESULTS.md successfully written to repository and Google Drive")


E8_RESULTS.md successfully written to repository and Google Drive


<>:44: SyntaxWarning: invalid escape sequence '\d'
<>:44: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_22177/1208122221.py:44: SyntaxWarning: invalid escape sequence '\d'
  """
